# REIT Cap Rate Spread Model
### Public Real Estate as an Asset Class — Project 1 of the Real Estate Track

**What this notebook does:**
Treats REITs as the liquid, tradeable proxy for real estate and builds the single most-watched
signal in real estate investing: the **spread between implied REIT yields and the 10-Year Treasury**.
When that spread is wide, real estate looks cheap relative to bonds. When it's narrow or negative,
real estate looks expensive — investors are accepting bond-like or worse compensation for illiquid,
levered property risk.

**Sectors covered:** Residential, Office, Industrial, Retail, Data Center, Healthcare

**Core outputs:**
1. Sector-level dividend yield and FFO-yield proxy (closer to a true "cap rate" than dividend yield alone)
2. Spread of each sector vs. the 10-Year Treasury, in basis points
3. A "rich vs. cheap" ranking across property sectors
4. Time series context for the 10Y Treasury so you can see the rate backdrop driving the spread

**Important framing for interviews:** public REIT yields are a *proxy* for private market cap rates,
not the same number Green Street or CoStar would report from actual property-level NOI data. This
notebook is explicit about that limitation rather than pretending otherwise — and that honesty about
limitations is exactly what a good interviewer wants to hear you articulate.


## 1. Setup

In [ ]:
!pip install -q yfinance plotly


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from io import StringIO
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Define the REIT universe

We organize REITs by property sector rather than treating "real estate" as one monolithic asset class —
office, industrial, and data center REITs have very different cash flow profiles and have diverged
sharply since 2020 (e.g., office under structural pressure, industrial/data center benefiting from
e-commerce and AI infrastructure demand). That divergence is exactly the kind of story this model
should surface.


In [ ]:
REIT_UNIVERSE = {
    "Residential": ["AVB", "EQR", "MAA", "ESS"],
    "Office": ["BXP", "VNO", "KRC"],
    "Industrial": ["PLD", "FR", "EGP"],
    "Retail": ["O", "SPG", "REG", "KIM"],
    "Data Center": ["DLR", "EQIX"],
    "Healthcare": ["WELL", "VTR", "DOC"],
}

BENCHMARK_ETF = "VNQ"      # broad public REIT market proxy
TREASURY_SERIES = "DGS10"  # FRED series ID: 10-Year Treasury Constant Maturity Rate

all_tickers = [tk for tks in REIT_UNIVERSE.values() for tk in tks]
print(f"Universe: {len(all_tickers)} REITs across {len(REIT_UNIVERSE)} sectors")


## 3. Pull the 10-Year Treasury yield (FRED)

FRED publishes most series as a public CSV with no API key required, which is the most reliable way
to do this inside Colab. If you later want richer macro context (CPI, Fed Funds, etc. for the Macro
Regime Dashboard project), get a free key at https://fred.stlouisfed.org/docs/api/api_key.html and
swap in the `fredapi` package — the CSV approach below is the fastest path for this project.


In [ ]:
def get_fred_series(series_id: str, start: str = "2015-01-01") -> pd.Series:
    """Pulls a FRED series via the public CSV endpoint - no API key required."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    df = pd.read_csv(StringIO(resp.text))
    df.columns = ["date", "value"]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna().set_index("date")
    return df.loc[df.index >= start, "value"]

treasury_10y = get_fred_series(TREASURY_SERIES)
current_10y = treasury_10y.iloc[-1]
print(f"Latest 10Y Treasury yield: {current_10y:.2f}%  (as of {treasury_10y.index[-1].date()})")
treasury_10y.tail()


In [ ]:
fig = px.line(
    treasury_10y,
    title="10-Year Treasury Yield (FRED: DGS10)",
    labels={"value": "Yield (%)", "date": "Date"},
)
fig.update_layout(showlegend=False, height=400)
fig.show()


## 4. Pull REIT-level data and build yield proxies

For each REIT we pull:
- **Dividend yield** (straightforward, from yfinance)
- **FFO-yield proxy** — Funds From Operations is the REIT-industry standard cash flow metric
  (Net Income + Depreciation & Amortization, since REIT depreciation is a non-cash accounting
  artifact that doesn't reflect real economic value loss on well-maintained property). We approximate
  it as `(Net Income + D&A) / Market Cap`. This is **not** the official NAREIT FFO figure — true FFO
  excludes gains on property sales and includes other REIT-specific adjustments not visible in a
  generic data feed — but it's a far better cap-rate proxy than dividend yield alone, and the gap
  between the two metrics is itself an interesting thing to show in an interview.


In [ ]:
def get_reit_snapshot(ticker: str) -> dict:
    t = yf.Ticker(ticker)
    info = t.info

    price = info.get("currentPrice") or info.get("regularMarketPrice")
    div_yield_raw = info.get("dividendYield")
    # yfinance has historically been inconsistent here: some versions return a
    # decimal fraction (0.045 = 4.5%), others return an already-converted percent
    # (4.5 = 4.5%). Detect and normalize so downstream math is always decimal.
    if div_yield_raw is not None and div_yield_raw > 1:
        div_yield = div_yield_raw / 100
    else:
        div_yield = div_yield_raw
    market_cap = info.get("marketCap")
    short_name = info.get("shortName")

    ffo_yield = np.nan
    try:
        cashflow = t.cashflow
        income = t.financials
        net_income = income.loc["Net Income"].iloc[0]
        dep_amort = cashflow.loc["Depreciation And Amortization"].iloc[0]
        ffo_approx = net_income + dep_amort
        if market_cap and market_cap > 0:
            ffo_yield = ffo_approx / market_cap
    except Exception:
        pass  # statement line item not found under this label for this ticker - leave NaN

    return {
        "ticker": ticker,
        "name": short_name,
        "price": price,
        "market_cap": market_cap,
        "dividend_yield": div_yield,
        "ffo_yield_proxy": ffo_yield,
    }


def build_universe_snapshot(universe: dict) -> pd.DataFrame:
    rows = []
    for sector, tickers in universe.items():
        for tk in tickers:
            snap = get_reit_snapshot(tk)
            snap["property_sector"] = sector
            rows.append(snap)
    return pd.DataFrame(rows)


snapshot_df = build_universe_snapshot(REIT_UNIVERSE)
snapshot_df


## 5. Compute spreads vs. the 10-Year Treasury

In [ ]:
def add_spreads(df: pd.DataFrame, current_10y: float) -> pd.DataFrame:
    out = df.copy()
    out["div_yield_pct"] = out["dividend_yield"] * 100
    out["ffo_yield_pct"] = out["ffo_yield_proxy"] * 100
    out["treasury_10y_pct"] = current_10y
    out["div_yield_spread_bps"] = (out["div_yield_pct"] - current_10y) * 100
    out["ffo_yield_spread_bps"] = (out["ffo_yield_pct"] - current_10y) * 100
    return out

spread_df = add_spreads(snapshot_df, current_10y)
spread_df.sort_values("div_yield_spread_bps", ascending=False)[
    ["ticker", "name", "property_sector", "div_yield_pct", "ffo_yield_pct",
     "div_yield_spread_bps", "ffo_yield_spread_bps"]
]


### Outlier check

Before trusting any sector average, check for individual REITs whose FFO-yield proxy
is wildly out of line with peers (e.g. >2x the sector median). This usually signals a
one-off accounting item (impairment, gain on sale, unusually large D&A relative to a
small market cap) rather than a genuine valuation signal — flag and consider excluding
it rather than letting it silently skew the sector average.


In [ ]:
sector_median_ffo = spread_df.groupby("property_sector")["ffo_yield_pct"].median()
spread_df["sector_median_ffo"] = spread_df["property_sector"].map(sector_median_ffo)
spread_df["ffo_outlier_flag"] = (
    spread_df["ffo_yield_pct"] > 2 * spread_df["sector_median_ffo"]
)
spread_df[spread_df["ffo_outlier_flag"]][
    ["ticker", "name", "property_sector", "ffo_yield_pct", "sector_median_ffo"]
]


## 6. Sector-level summary

This is the table you'd actually show in an interview or put on a slide — individual REIT noise
averaged out, sector signal preserved.


In [ ]:
def sector_summary(df: pd.DataFrame) -> pd.DataFrame:
    agg = df.groupby("property_sector").agg(
        avg_div_yield_pct=("div_yield_pct", "mean"),
        avg_ffo_yield_pct=("ffo_yield_pct", "mean"),
        avg_div_spread_bps=("div_yield_spread_bps", "mean"),
        avg_ffo_spread_bps=("ffo_yield_spread_bps", "mean"),
        n_reits=("ticker", "count"),
    ).sort_values("avg_div_spread_bps", ascending=False)
    return agg.round(2)

summary_df = sector_summary(spread_df)
summary_df


## 7. Visualizations

In [ ]:
fig = go.Figure()
fig.add_bar(
    x=summary_df.index,
    y=summary_df["avg_div_yield_pct"],
    name="Avg Dividend Yield (%)",
)
fig.add_hline(
    y=current_10y,
    line_dash="dash",
    line_color="red",
    annotation_text=f"10Y Treasury: {current_10y:.2f}%",
    annotation_position="top right",
)
fig.update_layout(
    title="Average REIT Dividend Yield by Sector vs. 10-Year Treasury",
    yaxis_title="Yield (%)",
    xaxis_title="Property Sector",
    height=450,
)
fig.show()


In [ ]:
fig = px.bar(
    summary_df.reset_index().sort_values("avg_div_spread_bps"),
    x="avg_div_spread_bps",
    y="property_sector",
    orientation="h",
    color="avg_div_spread_bps",
    color_continuous_scale="RdYlGn",
    title="Dividend Yield Spread vs. 10Y Treasury by Sector (bps)<br><sup>Wider/greener = cheaper relative to bonds | narrower/redder = more expensive</sup>",
    labels={"avg_div_spread_bps": "Spread (bps)", "property_sector": "Sector"},
)
fig.update_layout(height=450, coloraxis_showscale=False)
fig.show()


In [ ]:
fig = px.scatter(
    spread_df,
    x="div_yield_pct",
    y="ffo_yield_pct",
    color="property_sector",
    size="market_cap",
    hover_name="name",
    text="ticker",
    title="Dividend Yield vs. FFO-Yield Proxy by REIT<br><sup>Points above the diagonal: FFO yield exceeds dividend yield (lower payout ratio / more retained cash flow)</sup>",
    labels={"div_yield_pct": "Dividend Yield (%)", "ffo_yield_pct": "FFO-Yield Proxy (%)"},
)
max_val = max(spread_df["div_yield_pct"].max(), spread_df["ffo_yield_pct"].max()) * 1.1
fig.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val, line=dict(dash="dash", color="gray"))
fig.update_traces(textposition="top center")
fig.update_layout(height=550)
fig.show()


## 8. Interpretation template

Use this section to write your own read of the current snapshot — this is what turns the notebook
into an actual research note rather than just code output. A few prompts to answer once you've run
the live data:

- **Which sector currently has the widest spread vs. Treasuries, and does that match the macro
  story you'd expect** (e.g., office wide because of structural demand concerns, industrial/data
  center narrow because of strong secular demand)?
- **Where does the FFO-yield proxy diverge most from the dividend yield?** A REIT with FFO yield
  well above dividend yield is retaining more cash flow (lower payout ratio) — worth flagging as
  potentially under-distributing relative to peers, or building capital for growth.
- **How has the spread moved as the Treasury yield has moved?** If REIT yields haven't risen as
  fast as Treasuries, public real estate has gotten *relatively* more expensive even if absolute
  REIT yields look high historically.

## 9. Limitations (state these explicitly in any write-up)

- Dividend yield and the FFO-yield proxy are **public market** signals, not true property-level cap
  rates (NOI / property value) — private market cap rates from Green Street, CoStar, or NCREIF can
  diverge from public REIT pricing, sometimes significantly, especially at turning points in the cycle.
- The FFO-yield proxy is a simplification (`Net Income + D&A`) and does not back out gains on property
  sales or apply the formal NAREIT FFO adjustments — treat it as directional, not precise.
- `yfinance` field availability varies by ticker and can change without notice; some tickers may return
  `NaN` for the FFO proxy if statement line items aren't labeled consistently — that's expected and
  worth explaining if asked, not a bug to hide.

## 10. Natural extensions

- Pull this same spread **historically** (quarterly snapshots over 5-10 years) to see how it behaved
  through the 2020 crash, the 2022 rate hiking cycle, and today — turns a snapshot into a time series story.
- Feed the sector classifications and spread output into the **Macro Regime Dashboard** project to test
  whether REIT sector spreads behave differently across growth/inflation regimes.
- Use the spread level as a top-down "is real estate cheap or expensive right now" input into the
  **RE Underwriting Model's** exit cap rate assumption — connecting public market pricing to a private
  deal's terminal value is exactly the kind of cross-market thinking that's compelling in interviews.


## 11. Payout ratio and cross-sector ranking

Sector averages hide a lot. A single cross-sector ranking — ignoring sector labels —
surfaces individual names that are mispriced relative to *everyone*, not just their
sector peers. We also add the **payout ratio** (`dividend yield / FFO-yield proxy`),
which tells you what share of generated cash flow is actually being distributed —
the key sustainability check from the benchmark table (under ~75% is healthy, above
~90-100% is a flag).


In [ ]:
spread_df["payout_ratio_pct"] = (
    spread_df["div_yield_pct"] / spread_df["ffo_yield_pct"] * 100
)

ranked = spread_df.sort_values("ffo_yield_spread_bps", ascending=False)[
    ["ticker", "name", "property_sector", "ffo_yield_pct",
     "ffo_yield_spread_bps", "payout_ratio_pct"]
].reset_index(drop=True)
ranked.index = ranked.index + 1  # rank starting at 1
ranked


In [ ]:
fig = px.bar(
    ranked,
    x="ffo_yield_spread_bps",
    y="ticker",
    orientation="h",
    color="property_sector",
    hover_data=["name", "payout_ratio_pct"],
    title="All 19 REITs Ranked by FFO-Yield Spread vs. 10Y Treasury (bps)"
          "<br><sup>Ignoring sector grouping reveals individual outliers sector averages hide</sup>",
    labels={"ffo_yield_spread_bps": "Spread vs. 10Y Treasury (bps)", "ticker": ""},
)
fig.update_layout(height=650, yaxis={"categoryorder": "total ascending"})
fig.show()


**What to look for:** check whether names from the *same* sector cluster together or
scatter across the ranking. Tight clustering (e.g. Residential) means the market broadly
agrees on that sector's risk/value. Wide scatter within one sector label (e.g. Healthcare,
Office) means the sector-level average is hiding two genuinely different stories — name
the specific tickers driving each end, don't just report the sector number.
